# Honest Full-Precision Eval — Path B

**Цель**: устранить inference-artifact в сравнении base vs GSPO vs KTO. Все три модели прогоняются через Hugging Face transformers в bf16 с **identical decoding protocol** + **identical Combined Judge** (Cerebras).

**Compute**: RTX 6000 Ada (48GB), ~8.71 units/hour. Estimated: 3 models × 143 calc problems × ~30s = ~3.6h = ~31 units (≪ 600).

**Output**: `evaluation/reports/honest_full_precision_2026-04-30.json` со строгим apple-to-apple сравнением.

**Структура**:
1. Setup (paths, imports, .env)
2. **DECODING_CONFIG** — TODO(human): главное методологическое решение
3. Load eval dataset (143 calc problems)
4. Per-model inference function (base / +GSPO adapter / +KTO adapter)
5. Combined Judge (Cerebras, единый для всех трёх)
6. Run + save

In [ ]:
# Cell 2: Setup (Colab edition)
# ─── GPU check (VRAM-based, robust к названию) ─────────────────
# 9B bf16 = ~22GB weights + KV cache + activations → need ≥24GB VRAM minimum.
# 96GB GPUs (RTX PRO 6000 Blackwell, A100 80GB) дают comfortable headroom для
# co-located scoring (Skywork-PRM-1.5B + tutor 9B).
import subprocess
gpu_info = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode().strip()
print(f'GPU: {gpu_info}')
gpu_name, memory_str = gpu_info.split(',', 1)
gpu_memory_mib = int(memory_str.strip().split()[0])
assert gpu_memory_mib >= 24_000, (
    f'Insufficient VRAM for 9B bf16: {gpu_memory_mib} MiB. Need >=24GB. '
    f'Switch runtime to A100 / L4 24GB+ / RTX 6000 / RTX PRO 6000 / L40 / A40.'
)
print(f'VRAM check passed: {gpu_memory_mib} MiB ({gpu_memory_mib/1024:.1f} GiB)')

# ─── Mount Drive (for .env + report persistence across sessions) ─────
from google.colab import drive
drive.mount('/content/drive')

# ─── Clone repo + install ─────────────────────────────────────
# NB: GitHub repo is Siesher/MIST (historical typo from MITS — the URL stuck).
import os
if not os.path.exists('/content/MITS'):
    !git clone https://github.com/Siesher/MIST.git /content/MITS
%cd /content/MITS
!git checkout 019-ns-vstar-dpo && git pull origin 019-ns-vstar-dpo

# Удаляем torchvision: text-only inference, его не используем.
# Без удаления transformers >=4.55 пытается импортировать torchvision.io в
# processing_utils, и при CUDA-version mismatch (Colab часто имеет torch CUDA13
# vs torchvision CUDA12.8) ломает весь HF stack каскадом.
!pip uninstall -y torchvision 2>/dev/null || true

# Latest unpinned transformers нужен для qwen3_5 model_type (Qwen3.5 — март 2026).
# `--upgrade-strategy eager` форсирует upgrade всех HF deps вместе, иначе старый
# huggingface_hub сохраняется и ломает import `is_offline_mode`.
!pip install -q -U --upgrade-strategy eager \
    transformers huggingface_hub tokenizers accelerate peft bitsandbytes \
    datasets trl openai python-dotenv

# ─── Hybrid-attention speedups for Qwen3.5 ────────────────────
# Qwen3.5 hybrid blocks (GDN linear-attention + causal conv1d) need these libs.
# Без них transformers warn "fast path is not available" → torch fallback ~2-3x slower.
# fla — Triton-based, no CUDA toolkit needed (works в Colab по умолчанию).
# causal-conv1d — native CUDA, pre-built wheel обычно подтянется автоматически.
print('Installing hybrid-attention speedups (Qwen3.5 fast path)...')
!pip install -q flash-linear-attention 2>&1 | tail -3
import subprocess as _sp
_r = _sp.run(['pip', 'install', '-q', 'causal-conv1d>=1.4.0', '--no-build-isolation'],
             capture_output=True, text=True, timeout=600)
if _r.returncode == 0:
    print('  causal-conv1d: installed (CUDA fast path enabled)')
else:
    _tail = (_r.stderr or _r.stdout or '')[-300:]
    print(f'  causal-conv1d: FAILED — falling back to torch (slower).\n    stderr tail: {_tail}')

# ─── HF auth (for private adapters; if Siesher/mits-qwen3-9b-gspo is public, skip) ─
from huggingface_hub import login as hf_login
# Or set HF_TOKEN in Drive .env — load_dotenv() ниже подхватит автоматически.

# ─── Stdlib + project imports ─────────────────────────────────
import sys, json, time, logging
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List

PROJECT_ROOT = Path('/content/MITS')
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import transformers
import huggingface_hub
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from dotenv import load_dotenv

# Sanity: версии должны быть compatible (transformers ≥ 4.50, hub ≥ 0.25)
print(f'transformers: {transformers.__version__} | huggingface_hub: {huggingface_hub.__version__} | torch: {torch.__version__}')

# Load .env from Drive (keeps Cerebras keys out of the repo + survives Colab restarts)
ENV_PATH = Path('/content/drive/MyDrive/MITS_secrets/.env')
if ENV_PATH.exists():
    load_dotenv(ENV_PATH)
    print(f'Loaded env from {ENV_PATH}')
    if os.environ.get('HF_TOKEN'):
        hf_login(token=os.environ['HF_TOKEN'])
        print('HF authenticated via .env')
else:
    print(f'No .env at {ENV_PATH}. Cerebras judge will fail — socratic scores=None.\n'
          f'    Create folder MITS_secrets/ on Drive and upload .env with CEREBRAS_API_KEY_1..10.')

from training.scripts.evaluate_stage import (
    SYSTEM_PROMPT_CALC,
    extract_answer,
    check_format_compliance,
    evaluate_combined_quality,
    load_eval_dataset,
)
from training.cerebras_client import CerebrasClient

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger('honest_eval')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
logger.info(f'Device: {DEVICE} | bf16: {torch.cuda.is_bf16_supported()}')

## Cell 3 — Decoding Configuration (зафиксирован)

Параметры применяются ОДИНАКОВО ко всем трём моделям (base, GSPO, KTO). Зафиксировано:

| Параметр | Значение | Обоснование |
|----------|----------|-------------|
| `num_predict` | 4096 | Покрывает 95-percentile thinking длин (GSPO учился с budget=2048; 4096 даёт запас на hard problems без overhead 8192). |
| `enable_thinking` | `True` | Матчит training distribution GSPO/KTO + native режим Qwen3.5-9B. False нивелировал бы RL-effect целиком — unfair. |
| `temperature` | 0.0 | Greedy для deterministic accuracy. Diversity sampling — Phase 1 (V-STaR). |
| `system_prompt` | `SYSTEM_PROMPT_CALC` | Apple-to-apple с предыдущими compare_base_vs_gspo отчётами. |

**Что это даёт для диплома**: section "Methodology — inference protocol" в одну таблицу. Альтернатива (запустить второй раз с `enable_thinking=False`) — рассматривается как _Appendix-grade ablation_, если останется compute после Phase 1-3.

In [ ]:
# Cell 4: Decoding config (filled — single-protocol run)
# Same config applied to all three models (base, GSPO, KTO).

DECODING_CONFIG = {
    # 2048: GSPO trained budget=2048, covers 95-percentile thinking. 4096 был для
    # запаса на hard problems, но удваивает время генерации. 2048 = 2x speedup без
    # значимой потери (truncation risk <5%). Жертва ради скорости в lean-demo.
    'num_predict': 2048,
    # True: matches GSPO/KTO training distribution (chat_template_kwargs.enable_thinking
    # =True in grpo_qwen3_5_9b_(8).ipynb). Qwen3.5-9B base also natively supports <think>.
    # False would nullify the RL effect — unfair to GSPO/KTO. Keep one consistent mode.
    'enable_thinking': True,
    # 0.0: greedy decoding for accuracy eval. Diversity sampling is for V-STaR (Phase 1).
    'temperature': 0.0,
    # Same calc system prompt used in compare_base_vs_gspo_*.json — preserves apple-to-
    # apple with prior reports for the Cerebras-only path; only inference layer changes.
    'system_prompt': SYSTEM_PROMPT_CALC,
    'rationale': (
        'Single thinking-on protocol mirrors training distribution + production deployment. '
        '2048 budget covers GSPO training distribution. Greedy decoding for deterministic accuracy.'
    ),
}

assert all(v is not None for v in DECODING_CONFIG.values()), 'Fill in DECODING_CONFIG keys'
logger.info(f'Decoding config: {DECODING_CONFIG}')

In [ ]:
# Cell 5: Load eval dataset — calc subset only (numeric / latex_boxed)
EVAL_PATH = PROJECT_ROOT / 'training/data/eval_dataset.jsonl'
all_problems = load_eval_dataset(str(EVAL_PATH))
calc_problems = [p for p in all_problems if p.get('answer_type') in ('numeric', 'latex_boxed')]
logger.info(f'Total: {len(all_problems)} | Calc subset: {len(calc_problems)}')
# Sanity
from collections import Counter
logger.info(f'By domain: {Counter(p["domain"] for p in calc_problems)}')
logger.info(f'By difficulty: {Counter(p["difficulty"] for p in calc_problems)}')

In [ ]:
# Cell 6: Model loading + inference
BASE_MODEL_ID = 'Qwen/Qwen3.5-9B'  # matches BASE_MODEL in grpo_qwen3_5_9b_(8).ipynb
ADAPTERS = {
    'base': None,
    'gspo': 'Siesher/mits-qwen3-9b-gspo',
    'kto':  'Siesher/mits-qwen3-9b-kto',
}

def load_model_with_adapter(adapter_id: str | None):
    """Load Qwen3.5-9B base in bf16, optionally apply LoRA adapter and merge."""
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map='auto',
        trust_remote_code=True,
    )
    if adapter_id:
        model = PeftModel.from_pretrained(model, adapter_id)
        model = model.merge_and_unload()
        logger.info(f'Merged adapter {adapter_id}')
    model.eval()
    return model, tokenizer

def generate_one(model, tokenizer, prompt: str) -> str:
    messages = [
        {'role': 'system', 'content': DECODING_CONFIG['system_prompt']},
        {'role': 'user', 'content': prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=DECODING_CONFIG['enable_thinking'],
    )
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=DECODING_CONFIG['num_predict'],
            do_sample=DECODING_CONFIG['temperature'] > 0,
            temperature=max(DECODING_CONFIG['temperature'], 1e-5),
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    completion = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return completion

In [ ]:
# Cell 7: Eval loop (Phase 0a — fast_mode=True, programmatic correctness only)
# Phase 0b async judge — отдельным скриптом потом, когда Cerebras quota свободна.
import re

def _normalize_for_compare(s) -> str:
    """Normalize answer string for numeric/symbolic comparison.

    Accepts str | int | float | None. eval_dataset.jsonl держит numeric
    ground_truth как int/float (JSON не coerce-ит в строки), поэтому
    str(s) делается до strip().
    """
    if s is None:
        return ''
    s = str(s).strip()
    if not s:
        return ''
    if s.startswith('$') and s.endswith('$'):
        s = s[1:-1].strip()
    s = re.sub(r'\\text\{[^}]*\}', '', s)
    s = re.sub(r'\\mathrm\{[^}]*\}', '', s)
    # Russian decimal comma → dot, strip whitespace incl. nbsp
    s = s.replace(' ', '').replace(' ', '').replace(',', '.')
    # Strip trailing punctuation
    s = s.rstrip('.;,')
    return s


def is_numeric_correct(extracted, ground_truth, tolerance: float = 0.02) -> bool | None:
    """Programmatic correctness check.

    - For numeric strings: relative tolerance 2% or absolute < 0.001 if truth ≈ 0.
    - For non-numeric: case-insensitive normalized string match.
    - Returns None if can't determine (empty extracted or both unparseable).
    """
    e_norm = _normalize_for_compare(extracted)
    g_norm = _normalize_for_compare(ground_truth)
    if not e_norm or not g_norm:
        return None
    try:
        ef, gf = float(e_norm), float(g_norm)
        if abs(gf) < 1e-9:
            return abs(ef) < 0.001
        return abs(ef - gf) / abs(gf) < tolerance
    except ValueError:
        return e_norm.lower() == g_norm.lower()


def eval_stage(stage_name: str, adapter_id: str | None, problems: List[Dict],
               fast_mode: bool = True) -> Dict[str, Any]:
    """Evaluate one stage.

    Args:
        fast_mode: Phase 0a (True) — programmatic correctness only, saves raw
                   completions для Phase 0b (async Cerebras judge).
                   Phase 0 original (False) — синхронный Cerebras judge per problem
                   (упирается в rate limits, не рекомендуется).
    """
    logger.info(f'=== Stage: {stage_name} ({adapter_id or "base"}) | fast_mode={fast_mode} ===')
    model, tokenizer = load_model_with_adapter(adapter_id)
    cerebras = None if fast_mode else CerebrasClient()
    completions = []
    t0 = time.time()
    for i, p in enumerate(problems):
        t_start = time.time()
        completion = generate_one(model, tokenizer, p['prompt'])
        t_gen = time.time() - t_start
        # Per-problem timing для первых 3 — диагностика fast-path vs fallback скорости.
        if i < 3:
            n_tokens = len(tokenizer.encode(completion))
            logger.info(f'  [{i+1}/{len(problems)}] gen={t_gen:.1f}s | tokens={n_tokens} | tok/s={n_tokens/max(t_gen,0.01):.1f}')
        extracted = extract_answer(completion)
        fmt = check_format_compliance(completion)
        visible = completion.split('</think>')[-1].strip() if '</think>' in completion else completion

        if fast_mode:
            correct = is_numeric_correct(extracted, p['ground_truth'])
            socratic, no_leak = None, None
        else:
            judge = evaluate_combined_quality(p['prompt'], visible, p['ground_truth'], cerebras)
            correct = judge.get('is_correct')
            socratic = judge.get('socratic_score')
            no_leak = judge.get('no_answer_leak')

        completions.append({
            'idx': i, 'domain': p['domain'], 'difficulty': p['difficulty'],
            'truth': p['ground_truth'], 'extracted': extracted,
            'correct': correct,
            'socratic_score': socratic,
            'no_answer_leak': no_leak,
            'completion_text': visible,  # saved для Phase 0b async judge
            **fmt,
        })

        log_every = 5 if fast_mode else 10
        if (i + 1) % log_every == 0:
            elapsed = time.time() - t0
            valid = [c for c in completions if c['correct'] is not None]
            acc = sum(1 for c in valid if c['correct']) / max(len(valid), 1)
            logger.info(f'  {i+1}/{len(problems)} | acc={acc:.3f} ({len(valid)} judged) | {elapsed/60:.1f} min')

    del model; torch.cuda.empty_cache()
    valid = [c for c in completions if c['correct'] is not None]
    accuracy = sum(1 for c in valid if c['correct']) / max(len(valid), 1)
    socratic_vals = [c['socratic_score'] for c in completions if c.get('socratic_score') is not None]
    leak_vals = [c['no_answer_leak'] for c in completions if c.get('no_answer_leak') is not None]
    return {
        'stage': stage_name, 'adapter': adapter_id, 'n': len(completions),
        'n_judged': len(valid),
        'accuracy': accuracy,
        'avg_socratic': (sum(socratic_vals) / len(socratic_vals)) if socratic_vals else None,
        'leak_rate': (sum(1 for v in leak_vals if v < 2) / len(leak_vals)) if leak_vals else None,
        'mode': 'fast_programmatic' if fast_mode else 'full_cerebras_judge',
        'completions': completions,
    }

In [ ]:
# Cell 8: Run Phase 0a (fast_mode=True — programmatic correctness, no Cerebras).
# Phase 0b (async Cerebras judge over saved completions) — отдельный скрипт потом.
results = {}
for stage_name, adapter_id in ADAPTERS.items():
    results[stage_name] = eval_stage(stage_name, adapter_id, calc_problems, fast_mode=True)

report = {
    'protocol': 'honest_full_precision_phase0a',
    'mode': 'fast_programmatic',
    'note': (
        'Phase 0a: accuracy via programmatic numeric/string match (extract_answer vs '
        'ground_truth, 2% tolerance). socratic_score / leak_rate deferred to Phase 0b '
        '(async Cerebras judge over saved completions).'
    ),
    'timestamp': datetime.utcnow().isoformat(),
    'decoding_config': {k: v for k, v in DECODING_CONFIG.items() if k != 'system_prompt'},
    'system_prompt_hash': hash(DECODING_CONFIG['system_prompt']),
    'n_problems': len(calc_problems),
    'base': {k: v for k, v in results['base'].items() if k != 'completions'},
    'gspo': {k: v for k, v in results['gspo'].items() if k != 'completions'},
    'kto':  {k: v for k, v in results['kto'].items()  if k != 'completions'},
    'completions': {stage: r['completions'] for stage, r in results.items()},  # raw text сохранён
}

# Save to repo (so it can be committed) + Drive mirror (so it survives Colab teardown)
date_tag = datetime.utcnow().strftime('%Y%m%d')
out_repo  = PROJECT_ROOT / f'evaluation/reports/honest_full_precision_phase0a_{date_tag}.json'
out_drive = Path(f'/content/drive/MyDrive/MITS_secrets/honest_full_precision_phase0a_{date_tag}.json')
out_repo.parent.mkdir(parents=True, exist_ok=True)
out_repo.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
if out_drive.parent.exists():
    out_drive.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    logger.info(f'Saved (Drive mirror): {out_drive}')
logger.info(f'Saved (repo): {out_repo}')

print('\n=== Phase 0a — Honest accuracy (full-precision bf16, identical decoding, programmatic correctness) ===')
for stage in ['base', 'gspo', 'kto']:
    r = results[stage]
    print(f"{stage:5s}  acc={r['accuracy']:.3f} ({r['n_judged']}/{r['n']} judged)")
print('\nNote: socratic_score / leak_rate — Phase 0b (run scripts/score_phase0b_async.py later).')